In [1]:
%pip install -U langgraph langchain langchain-core langchain-google-genai python-dotenv
import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env file")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=api_key
)

print("Gemini initialized successfully!")


Note: you may need to restart the kernel to use updated packages.
Gemini initialized successfully!


In [7]:
class WorkflowState(TypedDict):
    user_input: str
    response: str
    status: str
    approved: bool
def process_request(state):
    print("\n[PROCESS] Processing request...")
    
    response = f"Processed request: {state['user_input']}"
    
    return {
        "response": response,
        "status": "waiting_for_approval"
    }
def human_approval(state):
    print("\n[APPROVAL] Workflow paused for human approval.")
    print("Response:", state["response"])
    
    decision = interrupt({
        "message": "Approve this response?",
        "response": state["response"]
    })
    
    return {
        "approved": decision == "approve",
        "status": "approved" if decision == "approve" else "rejected"
    }
def finish_workflow(state):
    if state["approved"]:
        print("\n[FINISH] Request approved.")
        return {
            "status": "completed"
        }
    
    print("\n[FINISH] Request rejected.")
    return {
        "status": "rejected"
    }
builder = StateGraph(WorkflowState)

builder.add_node("process", process_request)
builder.add_node("human_approval", human_approval)
builder.add_node("finish", finish_workflow)

builder.add_edge(START, "process")
builder.add_edge("process", "human_approval")
builder.add_edge("human_approval", "finish")
builder.add_edge("finish", END)

memory = MemorySaver()

graph = builder.compile(checkpointer=memory)


In [8]:
# Thread configuration for the workflow
config = {
    "configurable": {
        "thread_id": "task5-session-1"
    }
}

In [9]:
initial_state = {
    "user_input": "Prepare a summary of today's research.",
    "response": "",
    "status": "",
    "approved": False
}

result = graph.invoke(
    initial_state,
    config=config
)

print("\nGraph paused.")
print("Current state:")
print(result)


[PROCESS] Processing request...

[APPROVAL] Workflow paused for human approval.
Response: Processed request: Prepare a summary of today's research.

Graph paused.
Current state:
{'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'waiting_for_approval', 'approved': False, '__interrupt__': [Interrupt(value={'message': 'Approve this response?', 'response': "Processed request: Prepare a summary of today's research."}, id='9d170e7e23ca2d98924b5361234b21ee')]}


In [10]:
current_state = graph.get_state(config)

print("Persisted state:")
print(current_state.values)

Persisted state:
{'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'waiting_for_approval', 'approved': False}


In [12]:
resumed = graph.invoke(
    Command(resume="approve"),
    config=config
)

print("\nResumed workflow:")
print(resumed)


Resumed workflow:
{'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'completed', 'approved': True}


In [ ]:
# Get State History
history = list(graph.get_state_history(config))

print("Number of saved states:", len(history))

for i, checkpoint in enumerate(history):
    print("Checkpoint", i)
    print("State:", checkpoint.values)

Number of saved states: 5
Checkpoint 0
State: {'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'completed', 'approved': True}
Checkpoint 1
State: {'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'approved', 'approved': True}
Checkpoint 2
State: {'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'waiting_for_approval', 'approved': False}
Checkpoint 3
State: {'user_input': "Prepare a summary of today's research.", 'response': '', 'status': '', 'approved': False}
Checkpoint 4
State: {}


In [ ]:
# Detailed Debugging
for i, checkpoint in enumerate(history):

    print(f"CHECKPOINT {i}")
    print("Values:", checkpoint.values)
    print("Next:", checkpoint.next)
    print("Config:", checkpoint.config)

CHECKPOINT 0
Values: {'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'completed', 'approved': True}
Next: ()
Config: {'configurable': {'thread_id': 'task5-session-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac4c3-e543-6703-8003-64fe34ae2c9c'}}
CHECKPOINT 1
Values: {'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'approved', 'approved': True}
Next: ('finish',)
Config: {'configurable': {'thread_id': 'task5-session-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ac4c3-e53f-604b-8002-e7f9cad709c2'}}
CHECKPOINT 2
Values: {'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'waiting_for_approval', 'approved': False}
Next: ('human_approval',)
Config: {'configurable': {'thread_id': 'task5-session-1', 'checkpoint_ns': '', 'checkp

In [19]:
if len(history) > 1:
    previous_state = history[-1]
    
    print("Earlier checkpoint:")
    print(previous_state.values)
    print("\nNext node:")
    print(previous_state.next)
else:
    print("Not enough checkpoints for replay demonstration.")

Earlier checkpoint:
{}

Next node:
('__start__',)


In [20]:
same_config = {
    "configurable": {
        "thread_id": "task5-session-1"
    }
}

saved_state = graph.get_state(same_config)

print("Recovered session state:")
print(saved_state.values)

Recovered session state:
{'user_input': "Prepare a summary of today's research.", 'response': "Processed request: Prepare a summary of today's research.", 'status': 'completed', 'approved': True}


In [21]:
new_config = {
    "configurable": {
        "thread_id": "task5-session-2"
    }
}

In [22]:
new_result = graph.invoke(
    {
        "user_input": "Create a short project update.",
        "response": "",
        "status": "",
        "approved": False
    },
    config=new_config
)

print("Second session paused.")
print(new_result)


[PROCESS] Processing request...

[APPROVAL] Workflow paused for human approval.
Response: Processed request: Create a short project update.
Second session paused.
{'user_input': 'Create a short project update.', 'response': 'Processed request: Create a short project update.', 'status': 'waiting_for_approval', 'approved': False, '__interrupt__': [Interrupt(value={'message': 'Approve this response?', 'response': 'Processed request: Create a short project update.'}, id='0fae2f242910d5843140275834d8b1bb')]}


## Task 5 — Persistence & Debugging

LangGraph uses checkpointers to persist graph state for a specific thread. In this task, MemorySaver was used to store the workflow state, allowing a paused workflow to be resumed later using the same thread ID.

The workflow was paused at a human approval interrupt and then resumed using Command(resume="approve"). State history was inspected using get_state_history(), which provided previous checkpoints and execution information for debugging and replay analysis.

A new thread ID creates an independent conversation state, while reusing an existing thread ID allows the previously stored state to be recovered.